In [1]:
!pip install -q jax jaxlib flax optax tensorflow-datasets matplotlib


In [ ]:
# Forward-Forward Algorithm in JAX
# -----------------------------------------------
#
# Install dependencies (uncomment if needed)
# !pip install -q jax jaxlib flax optax tensorflow-datasets matplotlib

import jax
import jax.numpy as jnp
from jax import random, jit, grad
import optax
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

# 1. Data Loading and Preprocessing
# Load full train/test sets as single batches (to mirror PyTorch code)
ds_train = tfds.load('mnist', split='train', batch_size=-1, as_supervised=True)
ds_test  = tfds.load('mnist', split='test',  batch_size=-1, as_supervised=True)

# Convert to NumPy
train_ds = tfds.as_numpy(ds_train)
test_ds  = tfds.as_numpy(ds_test)

# Extract images and labels
train_images, train_labels = train_ds
test_images,  test_labels  = test_ds

# Normalize and flatten
x_train = jnp.float32(train_images) / 255.0
x_train = (x_train - 0.1307) / 0.3081
x_train = x_train.reshape((x_train.shape[0], -1))  # [60000, 784]

y_train = jnp.array(train_labels)                  # [60000]

x_test = jnp.float32(test_images) / 255.0
x_test = (x_test - 0.1307) / 0.3081
x_test = x_test.reshape((x_test.shape[0], -1))     # [10000, 784]

y_test = jnp.array(test_labels)                    # [10000]

# 2. Overlay Labels into Inputs
def overlay_y_on_x(x, y):
    # x: [batch, 784], y: [batch]
    max_val = jnp.max(x, axis=1, keepdims=True)
    x_mod = x.at[:, :10].set(0.0)
    idx = jnp.arange(x.shape[0])
    x_mod = x_mod.at[idx, y].set(max_val.flatten())
    return x_mod

# 3. Layer Definition
def init_layer_params(rng, in_dim, out_dim):
    k1, _ = random.split(rng)
    w = random.normal(k1, (in_dim, out_dim)) * jnp.sqrt(2.0 / in_dim)
    b = jnp.zeros((out_dim,))
    return {'w': w, 'b': b}

@jit
def forward_layer(params, x):
    normed = x / (jnp.linalg.norm(x, axis=1, keepdims=True) + 1e-4)
    preact = jnp.dot(normed, params['w']) + params['b']
    return jnp.maximum(preact, 0)

# 4. Prediction
def predict(all_params, x):
    batch = x.shape[0]
    scores = []
    for lbl in range(10):
        x_lab = overlay_y_on_x(x, jnp.full((batch,), lbl))
        h = x_lab
        total = jnp.zeros((batch,))
        for p in all_params:
            h = forward_layer(p, h)
            total += jnp.mean(h**2, axis=1)
        scores.append(total)
    preds = jnp.stack(scores, axis=1)  # [batch, 10]
    return jnp.argmax(preds, axis=1)

# 5. Training Function for One Layer
def train_layer(params, opt_state, x_pos, x_neg, threshold=2.0, epochs=1000, lr=0.03):
    optimizer = optax.adam(lr)

    @jit
    def loss_fn(p, xp, xn):
        gp = jnp.mean(forward_layer(p, xp)**2, axis=1)
        gn = jnp.mean(forward_layer(p, xn)**2, axis=1)
        lp = -gp + threshold
        ln = gn - threshold
        return jnp.mean(jnp.log1p(jnp.exp(jnp.concatenate([lp, ln]))))

    @jit
    def update_fn(p, opt_st, xp, xn):
        grads = grad(loss_fn)(p, xp, xn)
        updates, opt_st = optimizer.update(grads, opt_st)
        p = optax.apply_updates(p, updates)
        return p, opt_st

    for epoch in range(epochs):
        params, opt_state = update_fn(params, opt_state, x_pos, x_neg)
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, loss {loss_fn(params, x_pos, x_neg):.4f}")
    h_pos = forward_layer(params, x_pos)
    h_neg = forward_layer(params, x_neg)
    return params, opt_state, h_pos, h_neg

# 6. End-to-End Training
key = random.PRNGKey(0)
layer_dims = [784, 500, 500]
all_params, all_opts = [], []

x_pos = overlay_y_on_x(x_train, y_train)
perm = random.permutation(key, x_train.shape[0])
x_neg = overlay_y_on_x(x_train, y_train[perm])

for i in range(len(layer_dims)-1):
    key, sk = random.split(key)
    p = init_layer_params(sk, layer_dims[i], layer_dims[i+1])
    all_params.append(p)
    all_opts.append(optax.adam(0.03).init(p))

h_pos, h_neg = x_pos, x_neg
for i, (p, opt_st) in enumerate(zip(all_params, all_opts)):
    print(f"Training layer {i}...")
    p, opt_st, h_pos, h_neg = train_layer(p, opt_st, h_pos, h_neg)
    all_params[i], all_opts[i] = p, opt_st

# 7. Compute Errors
train_pred = predict(all_params, x_train)
print("Train error:", float(1 - jnp.mean(train_pred == y_train)))
test_pred  = predict(all_params, x_test)
print("Test error:",  float(1 - jnp.mean(test_pred  == y_test)))

# 8. Visualization
plt.figure(figsize=(8,4))
for idx, (data, name) in enumerate([(x_train, 'orig'), (x_pos, 'pos'), (x_neg, 'neg')]):
    plt.subplot(1,3,idx+1)
    img = jnp.reshape(data[0], (28,28))
    plt.title(name)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
plt.show()


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.BMRTFE_3.0.1/mnist-train.tfrecord*...:   0%|          | 0…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.BMRTFE_3.0.1/mnist-test.tfrecord*...:   0%|          | 0/…

Dataset mnist downloaded and prepared to /root/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.
